In [ ]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1


dataset_name = "pong"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 32
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 128
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)
target_return=10

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=batch_size,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=100#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=64, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=context_size,
    num_heads=4,#8
    num_layers=3,#6
    max_timestep=2000,
    compile_graph=False,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=100000,# 100000,
    n_steps_per_epoch=1000,# 1000,
    #save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=10
)

2025-07-29 16:12.00 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-29 16:12.00 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-29 16:12.00 [info     ] Action size has been automatically determined. action_size=6
2025-07-29 16:12.00 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-07-29 16:12.00 [debug    ] Building models...            
2025-07-29 16:12.01 [debug    ] Models have been built.       
2025-07-29 16:1

Epoch 1/100: 100%|██████████| 1000/1000 [09:41<00:00,  1.72it/s, actor_loss=1.32, critic_loss=0.0113]


2025-07-29 16:22.00 [info     ] New best score                 epoch=1 score=-21.0
2025-07-29 16:22.00 [info     ] Discrete_TACR_pong_1_20250729161201: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.10629625868797302, 'time_algorithm_update': 0.4744853546619415, 'actor_loss': 1.313913816988468, 'critic_loss': 0.0112730173910968, 'time_step': 0.580916425704956, 'eval_episode_mean_reward': -21.0, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -21.0, 'eval_episode_count': 1.0} step=1000
2025-07-29 16:22.01 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_TACR_pong_1_20250729161201/model_epoch_1.d3


Epoch 2/100: 100%|██████████| 1000/1000 [05:38<00:00,  2.96it/s, actor_loss=0.746, critic_loss=0.00702]

2025-07-29 16:27.39 [info     ] Discrete_TACR_pong_1_20250729161201: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.1047300295829773, 'time_algorithm_update': 0.23258860278129578, 'actor_loss': 0.7440656957030296, 'critic_loss': 0.007018192521296441, 'time_step': 0.33744830584526064} step=2000


2025-07-29 16:27.39 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_TACR_pong_1_20250729161201/model_epoch_2.d3


Epoch 3/100: 100%|██████████| 1000/1000 [05:35<00:00,  2.98it/s, actor_loss=0.317, critic_loss=0.00577]

2025-07-29 16:33.15 [info     ] Discrete_TACR_pong_1_20250729161201: epoch=3 step=3000 epoch=3 metrics={'time_sample_batch': 0.10372429084777832, 'time_algorithm_update': 0.23099129199981688, 'actor_loss': 0.31530225759744646, 'critic_loss': 0.0057678519894834605, 'time_step': 0.3348478231430054} step=3000


2025-07-29 16:33.15 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_TACR_pong_1_20250729161201/model_epoch_3.d3


Epoch 4/100:  14%|█▍        | 140/1000 [00:47<04:55,  2.91it/s, actor_loss=0.152, critic_loss=0.00556]

Test on A100

In [3]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1



dataset_name = "pong"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 128
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 128
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=batch_size,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=64, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=context_size,
    num_heads=4,#8
    num_layers=3,#6
    max_timestep=2000,
    compile_graph=False,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=100000,# 100000,
    n_steps_per_epoch=1000,# 1000,
    #save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=10#10
)

2025-08-06 09:53.34 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-08-06 09:53.34 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-08-06 09:53.34 [info     ] Action size has been automatically determined. action_size=6
2025-08-06 09:53.34 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-08-06 09:53.34 [debug    ] Building models...            
2025-08-06 09:53.34 [debug    ] Models have been built.       
2025-08-06 09:5

Epoch 1/100: 100%|██████████| 1000/1000 [22:08<00:00,  1.33s/it, actor_loss=1.26, critic_loss=0.0105]


2025-08-06 10:15.47 [info     ] New best score                 epoch=1 score=-21.0
2025-08-06 10:15.47 [info     ] Saving model 'd3rlpy_logs/Discrete_TACR_pong_1_20250806095334/model_epoch_1.d3' epoch=1
2025-08-06 10:15.47 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_TACR_pong_1_20250806095334/model_epoch_1.d3
2025-08-06 10:15.47 [info     ] Discrete_TACR_pong_1_20250806095334: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.4626523048877716, 'time_algorithm_update': 0.8648806056976318, 'actor_loss': 1.25576224565506, 'critic_loss': 0.010516572738531977, 'time_step': 1.327670024394989, 'eval_episode_mean_reward': -21.0, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -21.0, 'eval_episode_count': 1.0} step=1000


Epoch 2/100: 100%|██████████| 1000/1000 [22:07<00:00,  1.33s/it, actor_loss=1.15, critic_loss=0.00606]

2025-08-06 10:37.55 [info     ] Discrete_TACR_pong_1_20250806095334: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.46033437848091124, 'time_algorithm_update': 0.8663356947898865, 'actor_loss': 1.1449193782806397, 'critic_loss': 0.006054003354627639, 'time_step': 1.3268053390979766} step=2000



Epoch 3/100: 100%|██████████| 1000/1000 [21:16<00:00,  1.28s/it, actor_loss=1.07, critic_loss=0.00526]

2025-08-06 10:59.12 [info     ] Discrete_TACR_pong_1_20250806095334: epoch=3 step=3000 epoch=3 metrics={'time_sample_batch': 0.4620123326778412, 'time_algorithm_update': 0.8138589022159577, 'actor_loss': 1.0659603475928308, 'critic_loss': 0.005257623125799, 'time_step': 1.2760102014541626} step=3000



Epoch 4/100:  59%|█████▉    | 590/1000 [12:32<08:42,  1.28s/it, actor_loss=0.995, critic_loss=0.00589]


KeyboardInterrupt: 

With the specifications above: 20 min per epoch training.
Would be interesting to see how much time evaluation takes here.

now with evaluation after every epoch and 100 instead of 1000:

In [ ]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')
seed = 1



dataset_name = "pong"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 128
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 128
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=batch_size,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=64, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=context_size,
    num_heads=4,#8
    num_layers=3,#6
    max_timestep=2000,
    compile_graph=False,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=10000,# 100000,
    n_steps_per_epoch=100,# 1000,
    #save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=1#10
)

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-08-06 11:39.06 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-08-06 11:39.06 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-08-06 11:39.06 [info     ] Action size has been automatically determined. action_size=6
2025-08-06 11:39.06 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=6)
2025-08-06 11:39.06 [debug    ] Building models...            
2025-08-06 11:39.07 [debug    ] Models have been built.       
2025-08-06 11:3

Epoch 1/100: 100%|██████████| 100/100 [02:12<00:00,  1.33s/it, actor_loss=1.28, critic_loss=0.0143]


2025-08-06 11:41.24 [info     ] New best score                 epoch=1 score=-21.0
2025-08-06 11:41.24 [info     ] Saving model 'd3rlpy_logs/Discrete_TACR_pong_1_20250806113907/model_epoch_1.d3' epoch=1
2025-08-06 11:41.24 [info     ] Model parameters are saved to d3rlpy_logs/Discrete_TACR_pong_1_20250806113907/model_epoch_1.d3
2025-08-06 11:41.24 [info     ] Discrete_TACR_pong_1_20250806113907: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.4581385946273804, 'time_algorithm_update': 0.8665217900276184, 'actor_loss': 1.2724597334861756, 'critic_loss': 0.01423160837031901, 'time_step': 1.324790232181549, 'eval_episode_mean_reward': -21.0, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -21.0, 'eval_episode_count': 1.0} step=100


Epoch 2/100: 100%|██████████| 100/100 [02:12<00:00,  1.33s/it, actor_loss=1.24, critic_loss=0.0135]


...

Epoch 47/100: 100%|██████████| 100/100 [02:11<00:00,  1.32s/it, actor_loss=0.791, critic_loss=0.0071]


2025-08-06 16:24.33 [info     ] Discrete_TACR_pong_1_20250806113907: epoch=47 step=4700 epoch=47 metrics={'time_sample_batch': 0.4581873679161072, 'time_algorithm_update': 0.8601551151275635, 'actor_loss': 0.7902280116081237, 'critic_loss': 0.007136388863436878, 'time_step': 1.318463418483734, 'eval_episode_mean_reward': -20.0, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 1.2489995996796797, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -17.0, 'eval_episode_count': 50.0} step=4700


Epoch 48/100: 100%|██████████| 100/100 [02:12<00:00,  1.33s/it, actor_loss=0.772, critic_loss=0.00753]


2025-08-06 16:30.52 [info     ] Discrete_TACR_pong_1_20250806113907: epoch=48 step=4800 epoch=48 metrics={'time_sample_batch': 0.46299473762512205, 'time_algorithm_update': 0.8628824806213379, 'actor_loss': 0.7720515364408493, 'critic_loss': 0.007507199719548226, 'time_step': 1.3259932279586792, 'eval_episode_mean_reward': -20.28, 'eval_episode_median_reward': -21.0, 'eval_episode_std_reward': 0.9806120537705012, 'eval_episode_min_reward': -21.0, 'eval_episode_max_reward': -16.0, 'eval_episode_count': 50.0} step=4800


Epoch 49/100:  50%|█████     | 50/100 [01:06<01:06,  1.32s/it, actor_loss=0.757, critic_loss=0.00802]

In [ ]:
now check whether other atari models are similarly slow:

In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/breakout/expert-v0')
seed = 2



dataset_name = "breakout"


# settings in original DT paper
# Batch size 512 Pong 128 Breakout, Qbert, Seaquest Context length K 50 Pong 30 Breakout, Qbert, Seaquest Return-to-go conditioning 90 Breakout (≈ 1× max in dataset) 2500 Qbert (≈ 5× max in dataset) 20 Pong (≈ 1× max in dataset) 1450 Seaquest (≈ 5× max in dataset)

if dataset_name == "pong":
    batch_size = 128
    context_size = 50
    target_return = 20
elif dataset_name == "breakout":
    batch_size = 32
    context_size = 30
    target_return = 90
elif dataset_name == "qbert":
    batch_size = 128
    context_size = 30
    target_return = 2500
elif dataset_name == "seaquest":    
    batch_size = 128
    context_size = 30
    target_return = 1450

# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=batch_size,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.PixelEncoderFactory(
            feature_size=64, exclude_last_activation=True
        ),
    observation_scaler=d3rlpy.preprocessing.PixelObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.GLOBAL,
    context_size=context_size,
    num_heads=4,#4 for pong#8 in DT
    num_layers=3,#3 for pong#6 in DT
    max_timestep=2000,
    compile_graph=False,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=100000,# 100000,
    n_steps_per_epoch=1000,# 1000,
    #save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=1,
    eval_gaps=1#10
)

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-08-06 11:15.55 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-08-06 11:15.55 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-08-06 11:15.55 [info     ] Action size has been automatically determined. action_size=4
2025-08-06 11:15.55 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]), action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]), reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)]), action_space=<ActionSpace.DISCRETE: 2>, action_size=4)
2025-08-06 11:15.55 [debug    ] Building models...            
2025-08-06 11:15.55 [debug    ] Models have been built.       
2025-08-06 11:1

Epoch 1/100:   0%|          | 0/1000 [00:00<?, ?it/s]
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [3,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [4,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/aten/src/ATen/native/cuda/Indexing.cu:1553: indexSelectLargeIndex: block: [2,0,0], thread: [5,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
/pytorch/a

RuntimeError: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR

In [5]:
print("Max timestep in dataset:", max([len(ep.actions) for ep in dataset.episodes]))

Max timestep in dataset: 1962


In [4]:
for ep in dataset.episodes:
    print(ep.observations.shape)
    break

(2363, 3, 210, 160)


In [2]:
env.observation_space

Box(0, 255, (210, 160, 3), uint8)

In [3]:
type(env)

gymnasium.wrappers.common.TimeLimit

In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory
import numpy as np
local = False
if local:
    os.environ["D3RLPY_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/home/julian/programming_data/d3rlpy_data/minari_data"
else:
    # on cluster
    os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
    os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

#os.environ["CUDA_LAUNCH_BLOCKING"]="1"
import d3rlpy

dataset, env = d3rlpy.datasets.get_minari('atari/pong/expert-v0')

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


2025-07-29 10:55.40 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('uint8')], shape=[(3, 210, 160)]) reward_signature=Signature(dtype=[dtype('float64')], shape=[(1,)])
2025-07-29 10:55.40 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-29 10:55.40 [info     ] Action size has been automatically determined. action_size=6


In [3]:
env.observation_space

Box(0, 255, (210, 160, 3), uint8)

In [7]:
one,two = env.reset()

In [8]:
print(type(one))
print(type(two))

<class 'numpy.ndarray'>
<class 'dict'>


In [10]:
print((one.shape))
print((two.keys()))

(210, 160, 3)
dict_keys(['lives', 'episode_frame_number', 'frame_number'])


In [8]:
for ep in dataset.episodes:
    print(type(ep.observations))
    print(type(ep))
    break

<class 'numpy.ndarray'>
<class 'd3rlpy.dataset.components.Episode'>
